# Entraînement COCO Stuff en ligne — callback d'équilibre

On passe sur **toutes les images** du dataset COCO Stuff. Le callback stoppe l'envoi d'images quand un **équilibre** se crée, via 3 critères :

- **A. Variation des poids** : $\|W^{(t)} - W^{(t-1)}\|_F < \epsilon_W$
- **B. Dissipation de la surprise résiduelle** : $dS/dt \approx 0$ (plateau)
- **C. Stabilisation du flux Physarum** : $\|\Delta D_{ij}\| < \epsilon$ (connexions)

Arrêt **automatique à 1h** max. Des **visuels** (ce que voit le système + architecture) sont générés périodiquement.

## 0. Imports + config

In [1]:
# Entraînement COCO Stuff en ligne : callback d'équilibre + visuels
import os
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from recherche_agi import (AnchorNeurons, dynamic_k, EquilibriumCallback,
                           color_texture_features, visualize_input,
                           visualize_architecture, visualize_evolution)

MAX_DURATION = 3600   # 1h
PATCH_SIZE = 32
N_NEURONS = 400
names = {118:'sky',113:'road',127:'tree',129:'wall-brick',145:'grass',133:'water',
         116:'sea',120:'snow',105:'house',123:'streetlight',112:'railing',115:'sand',
         114:'roof',146:'dirt'}

c:\Users\henry\Desktop\workspace\recherche-agi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Initialisation

In [2]:
anchors = AnchorNeurons(d_in=35, n_neurons=N_NEURONS, seed=0, lr=0.05,
                          use_homeostasis=True)
cb = EquilibriumCallback(eps_W=1e-3, eps_S=1e-3, stable_window=15,
                         check_every=20, max_duration_s=MAX_DURATION)
print(f"AnchorNeurons : {N_NEURONS} neurones, d_in=35 (features texture+couleur)")
print("Callback d'équilibre initialisé (arrêt à 1h max)")

AnchorNeurons : 400 neurones, d_in=35 (features texture+couleur)
Callback d'équilibre initialisé (arrêt à 1h max)


## 2. Entraînement en ligne sur toutes les images

In [3]:
ds = load_dataset('shunk031/cocostuff', 'stuff-thing', split='train',
                  streaming=True, trust_remote_code=True)
n_img = 0; n_patch = 0; seen_class_patches = {}
acc_history = []; t_start = time.time()

for x in ds:
    n_img += 1
    try:
        img = np.array(x['image']); sm = np.array(x['stuff_map'])
        H, W = sm.shape
        patches_img = []
        for i in range(0, H-PATCH_SIZE, PATCH_SIZE):
            for j in range(0, W-PATCH_SIZE, PATCH_SIZE):
                ps = sm[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
                pim = img[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
                vals, counts = np.unique(ps, return_counts=True)
                if vals[np.argmax(counts)] == 255: continue
                cls = int(vals[np.argmax(counts)])
                patches_img.append((cls, pim))
        for cls, pim in patches_img:
            f = color_texture_features(pim)
            zn = f / (np.linalg.norm(f)+1e-8)
            anchors.learn(zn, k=dynamic_k(0.5,1,5), label=cls)
            sim = anchors.W @ zn; w = int(np.argmax(sim))
            S = float(np.linalg.norm(zn - anchors.W[w])**2)
            if cb.on_image(anchors.W.copy(), anchors.co_act.copy(), S):
                n_patch += 1
                break
            if cls not in seen_class_patches: seen_class_patches[cls] = pim
            n_patch += 1
    except Exception as e:
        print(f"  skip {n_img}: {str(e)[:50]}")

    if n_img % 30 == 0:
        print(f"  [{cb.elapsed():.0f}s] images={n_img} patches={n_patch} neurones={len(anchors.W)}", end='\r')
    if n_img % 40 == 0:
        # visuels périodiques inline
        _ = visualize_input({c:p[None] for c,p in seen_class_patches.items()},
                              names, n_cols=1, out='__tmp_input.png')
        print(f"\n  [visuel] {n_img} images traitées, {n_patch} patches")
    if cb.stopped_by:
        print(f"\n  ARRÊT : {cb.stopped_by}")
        break

s = cb.summary()
print(f"\n=== FIN : images={n_img}, patches={n_patch} ===")
print(f"  arrêt={s['stopped_by']}, durée={s['elapsed_s']:.1f}s")
print(f"  neurones={len(anchors.W)}")
# visuels finaux
fig1 = visualize_architecture(anchors, out='__tmp_arch.png')
fig2 = visualize_evolution(cb, out='__tmp_evo.png')
# afficher
from IPython.display import Image as IImg, display
print("\n--- Architecture du modèle ---")
display(IImg('__tmp_arch.png'))
print("\n--- Évolution des 3 critères d'équilibre ---")
display(IImg('__tmp_evo.png'))
print("\n--- Résumé callback ---")
print(s)

  [259s] images=30 patches=7328 neurones=400
  [visuel] 40 images traitées, 9856 patches
  [531s] images=60 patches=14916 neurones=400
  [visuel] 80 images traitées, 19782 patches
  [972s] images=120 patches=28454 neurones=400
  [visuel] 120 images traitées, 28454 patches
  [1192s] images=150 patches=35573 neurones=400
  [visuel] 160 images traitées, 38015 patches
  [1420s] images=180 patches=42713 neurones=400
  [visuel] 200 images traitées, 47435 patches
  [1946s] images=240 patches=56706 neurones=400
  [visuel] 240 images traitées, 56706 patches


KeyboardInterrupt: 

## 3. Analyse

In [4]:
print("=== ANALYSE : ENTRAÎNEMENT EN LIGNE COCO ===")
s = cb.summary()
print(f"1. Arrêt : {s['stopped_by']} après {s['elapsed_s']:.0f}s et {s['n_images']} images.")
print(f"2. Critères d'équilibre :")
print(f"   - ΔW final : {s['last_dW']:.2e} (A)")
print(f"   - ΔS final : {s['last_S']:.2e} (B)")
print(f"   - ΔD final : {s['last_D']:.2e} (C)")
print(f"3. Neurones formés : {len(anchors.W)}")
print()
print("=> Le callback stoppe l'envoi d'images quand l'équilibre se crée (ΔW, ΔS,")
print("   ΔD stabilisés) ou à 1h max. Les visuels montrent ce que le système voit")
print("   (échantillons de patches) et l'architecture (neurones, connectivité).")

=== ANALYSE : ENTRAÎNEMENT EN LIGNE COCO ===
1. Arrêt : None après 2380s et 65544 images.
2. Critères d'équilibre :
   - ΔW final : 4.24e-02 (A)
   - ΔS final : 3.82e-02 (B)
   - ΔD final : 2.45e+00 (C)
3. Neurones formés : 400

=> Le callback stoppe l'envoi d'images quand l'équilibre se crée (ΔW, ΔS,
   ΔD stabilisés) ou à 1h max. Les visuels montrent ce que le système voit
   (échantillons de patches) et l'architecture (neurones, connectivité).
